# Setup

*If running in Google Colab, go to Runtime > Change Runtime Type and select GPU as the hardware accelerator.*

### Environment Setup (ignore)

**You can ignore this part:** It's just for use internally to setup the tutorial in different
environments. You can delete this section if using in your own repo.

In [ ]:

# Detect if we're running in Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
except:
    IN_COLAB = False

# Install if in Colab
if IN_COLAB:
    %pip install transformer_lens
    %pip install circuitsvis
    # Install a faster Node version
    !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash -; sudo apt-get install -y nodejs  # noqa

# Hot reload in development mode & not running on the CD
if not IN_COLAB:
    from IPython import get_ipython
    ip = get_ipython()
    if not ip.extension_manager.loaded:
        ip.extension_manager.load('autoreload')
        %autoreload 2


Running as a Colab notebook


                              DEPRECATION WARNING                            

     Node.js 16.x is no longer actively supported!

  You will not receive security or critical stability updates for this version.

  You should migrate to a supported version of Node.js as soon as possible.
  Use the installation script that corresponds to the version of Node.js you
  wish to install. e.g.
  
   * https://deb.nodesource.com/setup_16.x — Node.js 16 "Gallium" (deprecated)
   * https://deb.nodesource.com/setup_18.x — Node.js 18 "Hydrogen" (Maintenance)
   * https://deb.nodesource.com/setup_19.x — Node.js 19 "Nineteen" (deprecated)
   * https://deb.nodesource.com/setup_20.x — Node.js 20 LTS "Iron" (recommended)
   * https://deb.nodesource.com/setup_21.x — Node.js 21 "Iron" (current)
   


  Please see https://github.com/nodejs/Release for details about which
  version may be appropriate for you.

  The NodeSource Node.js distributions repository contains
  informa

### Imports

In [ ]:
from functools import partial
from typing import List, Optional, Union

import einops
import numpy as np
import plotly.express as px
import plotly.io as pio
import torch
from circuitsvis.attention import attention_heads
from fancy_einsum import einsum
from IPython.display import HTML, IFrame
from jaxtyping import Float

import transformer_lens.utils as utils
from transformer_lens import ActivationCache, HookedTransformer

### PyTorch Setup

We turn automatic differentiation off, to save GPU memory, as this notebook focuses on model inference not model training.

In [ ]:
torch.set_grad_enabled(False)
print("Disabled automatic differentiation")

Disabled automatic differentiation


### Plotting Helper Functions (ignore)

Some plotting helper functions are included here (for simplicity).

In [ ]:
def imshow(tensor, **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        **kwargs,
    ).show()


def line(tensor, **kwargs):
    px.line(
        y=utils.to_numpy(tensor),
        **kwargs,
    ).show()


def scatter(x, y, xaxis="", yaxis="", caxis="", **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(
        y=y,
        x=x,
        labels={"x": xaxis, "y": yaxis, "color": caxis},
        **kwargs,
    ).show()

The first step is to load in our model, GPT-2 Small, a 12 layer and 80M parameter transformer with `HookedTransformer.from_pretrained`. The various flags are simplifications that preserve the model's output but simplify its internals.

In [ ]:
# NBVAL_IGNORE_OUTPUT
model = HookedTransformer.from_pretrained(
    "gpt2-small",
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    refactor_factored_attn_matrices=True,
)

# Get the default device used
device: torch.device = utils.get_device()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer


# Task

The next step is to verify that the model can *actually* do the task! Here we use `utils.test_prompt`, and see that the model is significantly better at predicting Mary than John!

<details><summary>Asides:</summary>

Note: If we were being careful, we'd want to run the model on a range of prompts and find the average performance

`prepend_bos` is a flag to add a BOS (beginning of sequence) to the start of the prompt. GPT-2 was not trained with this, but I find that it often makes model behaviour more stable, as the first token is treated weirdly.
</details>

In [ ]:
prompt = "After John and Mary went to the store, John gave a bottle of milk to"
answer = " Mary"   # Note the proceeding space to match the model's tokenization
utils.test_prompt(prompt, answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'After', ' John', ' and', ' Mary', ' went', ' to', ' the', ' store', ',', ' John', ' gave', ' a', ' bottle', ' of', ' milk', ' to']
Tokenized answer: [' Mary']


Performance on answer token:
Rank: 0        Logit: 18.09 Prob: 70.07% Token: | Mary|

Top 0th token. Logit: 18.09 Prob: 70.07% Token: | Mary|
Top 1th token. Logit: 15.38 Prob:  4.67% Token: | the|
Top 2th token. Logit: 15.35 Prob:  4.54% Token: | John|
Top 3th token. Logit: 15.25 Prob:  4.11% Token: | them|
Top 4th token. Logit: 14.84 Prob:  2.73% Token: | his|
Top 5th token. Logit: 14.06 Prob:  1.24% Token: | her|
Top 6th token. Logit: 13.54 Prob:  0.74% Token: | a|
Top 7th token. Logit: 13.52 Prob:  0.73% Token: | their|
Top 8th token. Logit: 13.13 Prob:  0.49% Token: | Jesus|
Top 9th token. Logit: 12.97 Prob:  0.42% Token: | him|


Ranks of the answer tokens: [(' Mary', 0)]

## Apple tree

In [ ]:
example_prompt = "The fruit of an apple tree is not the peach, but the"
example_answer = " apple"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' not', ' the', ' peach', ',', ' but', ' the']
Tokenized answer: [' apple']


Performance on answer token:
Rank: 0        Logit: 13.41 Prob:  4.94% Token: | apple|

Top 0th token. Logit: 13.41 Prob:  4.94% Token: | apple|
Top 1th token. Logit: 13.41 Prob:  4.90% Token: | fruit|
Top 2th token. Logit: 13.35 Prob:  4.62% Token: | peach|
Top 3th token. Logit: 12.94 Prob:  3.06% Token: | cherry|
Top 4th token. Logit: 12.60 Prob:  2.18% Token: | plum|
Top 5th token. Logit: 12.37 Prob:  1.74% Token: | pear|
Top 6th token. Logit: 12.29 Prob:  1.61% Token: | sweet|
Top 7th token. Logit: 12.19 Prob:  1.46% Token: | red|
Top 8th token. Logit: 12.05 Prob:  1.27% Token: | orange|
Top 9th token. Logit: 11.99 Prob:  1.18% Token: | tree|


Ranks of the answer tokens: [(' apple', 0)]

In [ ]:
example_prompt = "The fruit of an apple tree is the apple. So, the fruit of an apple tree is the"
example_answer = " apple"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' the', ' apple', '.', ' So', ',', ' the', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' the']
Tokenized answer: [' apple']


Performance on answer token:
Rank: 0        Logit: 17.08 Prob: 62.36% Token: | apple|

Top 0th token. Logit: 17.08 Prob: 62.36% Token: | apple|
Top 1th token. Logit: 16.23 Prob: 26.54% Token: | fruit|
Top 2th token. Logit: 13.48 Prob:  1.70% Token: | apples|
Top 3th token. Logit: 12.87 Prob:  0.93% Token: | tree|
Top 4th token. Logit: 12.35 Prob:  0.55% Token: | fruits|
Top 5th token. Logit: 11.18 Prob:  0.17% Token: | cherry|
Top 6th token. Logit: 11.18 Prob:  0.17% Token: | "|
Top 7th token. Logit: 11.08 Prob:  0.15% Token: | Apple|
Top 8th token. Logit: 10.99 Prob:  0.14% Token: | peach|
Top 9th token. Logit: 10.98 Prob:  0.14% Token: | root|


Ranks of the answer tokens: [(' apple', 0)]

In [ ]:
example_prompt = "The fruit of an apple tree is the apple. So, the fruit of an apple tree is not the peach, but the"
example_answer = " apple"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' the', ' apple', '.', ' So', ',', ' the', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' not', ' the', ' peach', ',', ' but', ' the']
Tokenized answer: [' apple']


Performance on answer token:
Rank: 0        Logit: 14.99 Prob: 17.20% Token: | apple|

Top 0th token. Logit: 14.99 Prob: 17.20% Token: | apple|
Top 1th token. Logit: 14.86 Prob: 15.07% Token: | fruit|
Top 2th token. Logit: 14.17 Prob:  7.61% Token: | peach|
Top 3th token. Logit: 13.01 Prob:  2.38% Token: | cherry|
Top 4th token. Logit: 12.96 Prob:  2.27% Token: | tree|
Top 5th token. Logit: 12.90 Prob:  2.12% Token: | pear|
Top 6th token. Logit: 12.53 Prob:  1.48% Token: | grape|
Top 7th token. Logit: 12.24 Prob:  1.10% Token: | plum|
Top 8th token. Logit: 12.17 Prob:  1.02% Token: | orange|
Top 9th token. Logit: 12.14 Prob:  1.00% Token: | red|


Ranks of the answer tokens: [(' apple', 0)]

In [ ]:
example_prompt = "The fruit of an apple tree is the apple. So, the apple can be the fruit of an apple tree?"
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' the', ' apple', '.', ' So', ',', ' the', ' apple', ' can', ' be', ' the', ' fruit', ' of', ' an', ' apple', ' tree', '?']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 4        Logit: 15.16 Prob:  3.73% Token: | Yes|

Top 0th token. Logit: 16.59 Prob: 15.67% Token: |
|
Top 1th token. Logit: 15.69 Prob:  6.33% Token: | Well|
Top 2th token. Logit: 15.48 Prob:  5.14% Token: | The|
Top 3th token. Logit: 15.31 Prob:  4.33% Token: | It|
Top 4th token. Logit: 15.16 Prob:  3.73% Token: | Yes|
Top 5th token. Logit: 15.12 Prob:  3.60% Token: | Or|
Top 6th token. Logit: 14.97 Prob:  3.10% Token: | That|
Top 7th token. Logit: 14.75 Prob:  2.47% Token: | In|
Top 8th token. Logit: 14.69 Prob:  2.34% Token: | And|
Top 9th token. Logit: 14.69 Prob:  2.33% Token: | No|


Ranks of the answer tokens: [(' Yes', 4)]

In [ ]:
example_prompt = "The fruit of an apple tree is the apple. So, the peach can be the fruit of an apple tree?"
example_answer = " No"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' the', ' apple', '.', ' So', ',', ' the', ' peach', ' can', ' be', ' the', ' fruit', ' of', ' an', ' apple', ' tree', '?']
Tokenized answer: [' No']


Performance on answer token:
Rank: 9        Logit: 14.67 Prob:  2.20% Token: | No|

Top 0th token. Logit: 16.74 Prob: 17.36% Token: |
|
Top 1th token. Logit: 15.73 Prob:  6.37% Token: | Well|
Top 2th token. Logit: 15.52 Prob:  5.15% Token: | The|
Top 3th token. Logit: 15.24 Prob:  3.91% Token: | Or|
Top 4th token. Logit: 15.19 Prob:  3.69% Token: | It|
Top 5th token. Logit: 15.14 Prob:  3.53% Token: | Yes|
Top 6th token. Logit: 15.05 Prob:  3.22% Token: | That|
Top 7th token. Logit: 14.73 Prob:  2.34% Token: | And|
Top 8th token. Logit: 14.70 Prob:  2.28% Token: | In|
Top 9th token. Logit: 14.67 Prob:  2.20% Token: | No|


Ranks of the answer tokens: [(' No', 9)]

In [ ]:
example_prompt = "The fruit of an apple tree is the apple. So, the peach can be the fruit of an apple tree? Yes or no?"
example_answer = " No"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' the', ' apple', '.', ' So', ',', ' the', ' peach', ' can', ' be', ' the', ' fruit', ' of', ' an', ' apple', ' tree', '?', ' Yes', ' or', ' no', '?']
Tokenized answer: [' No']


Performance on answer token:
Rank: 12       Logit: 14.28 Prob:  1.32% Token: | No|

Top 0th token. Logit: 16.96 Prob: 19.10% Token: |
|
Top 1th token. Logit: 15.68 Prob:  5.32% Token: | The|
Top 2th token. Logit: 15.49 Prob:  4.42% Token: | Well|
Top 3th token. Logit: 15.22 Prob:  3.35% Token: | It|
Top 4th token. Logit: 15.16 Prob:  3.17% Token: | I|
Top 5th token. Logit: 14.70 Prob:  2.01% Token: | Yes|
Top 6th token. Logit: 14.63 Prob:  1.87% Token: | And|
Top 7th token. Logit: 14.60 Prob:  1.81% Token: | If|
Top 8th token. Logit: 14.57 Prob:  1.75% Token: | In|
Top 9th token. Logit: 14.40 Prob:  1.48% Token: | That|


Ranks of the answer tokens: [(' No', 12)]

In [ ]:
example_prompt = "The apple, for an apple tree, is its own"
example_answer = " fruit"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' apple', ',', ' for', ' an', ' apple', ' tree', ',', ' is', ' its', ' own']
Tokenized answer: [' fruit']


Performance on answer token:
Rank: 5        Logit: 13.85 Prob:  2.64% Token: | fruit|

Top 0th token. Logit: 14.66 Prob:  5.95% Token: | kind|
Top 1th token. Logit: 14.36 Prob:  4.38% Token: | species|
Top 2th token. Logit: 14.23 Prob:  3.86% Token: | tree|
Top 3th token. Logit: 14.20 Prob:  3.74% Token: | thing|
Top 4th token. Logit: 14.01 Prob:  3.08% Token: | genus|
Top 5th token. Logit: 13.85 Prob:  2.64% Token: | fruit|
Top 6th token. Logit: 13.77 Prob:  2.43% Token: | little|
Top 7th token. Logit: 13.69 Prob:  2.25% Token: | food|
Top 8th token. Logit: 13.63 Prob:  2.12% Token: | unique|
Top 9th token. Logit: 13.55 Prob:  1.96% Token: | family|


Ranks of the answer tokens: [(' fruit', 5)]

MCQs

In [ ]:
example_prompt = "The fruit of an apple tree is not the peach, but the [a: peach, b: cherry, c: apple, d: tree]"
example_answer = " a"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'The', ' fruit', ' of', ' an', ' apple', ' tree', ' is', ' not', ' the', ' peach', ',', ' but', ' the', ' [', 'a', ':', ' peach', ',', ' b', ':', ' cherry', ',', ' c', ':', ' apple', ',', ' d', ':', ' tree', ']']
Tokenized answer: [' a']


Performance on answer token:
Rank: 20       Logit: 10.52 Prob:  0.65% Token: | a|

Top 0th token. Logit: 13.16 Prob:  9.17% Token: | fruit|
Top 1th token. Logit: 12.59 Prob:  5.18% Token: | tree|
Top 2th token. Logit: 12.57 Prob:  5.04% Token: |
|
Top 3th token. Logit: 12.56 Prob:  5.01% Token: | apple|
Top 4th token. Logit: 12.08 Prob:  3.10% Token: | and|
Top 5th token. Logit: 12.07 Prob:  3.07% Token: | [|
Top 6th token. Logit: 11.71 Prob:  2.13% Token: | (|
Top 7th token. Logit: 11.62 Prob:  1.96% Token: | of|
Top 8th token. Logit: 11.61 Prob:  1.94% Token: | is|
Top 9th token. Logit: 11.61 Prob:  1.93% Token: | peach|


Ranks of the answer tokens: [(' a', 20)]

## Capital

### Rome - Madrid

In [ ]:
example_prompt = "Rome is the capital of"
example_answer = " Italy"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'R', 'ome', ' is', ' the', ' capital', ' of']
Tokenized answer: [' Italy']


Performance on answer token:
Rank: 0        Logit: 16.16 Prob: 28.04% Token: | Italy|

Top 0th token. Logit: 16.16 Prob: 28.04% Token: | Italy|
Top 1th token. Logit: 16.05 Prob: 25.17% Token: | the|
Top 2th token. Logit: 14.13 Prob:  3.69% Token: | Rome|
Top 3th token. Logit: 13.94 Prob:  3.04% Token: | a|
Top 4th token. Logit: 13.71 Prob:  2.44% Token: | Romania|
Top 5th token. Logit: 13.47 Prob:  1.92% Token: | Europe|
Top 6th token. Logit: 13.20 Prob:  1.45% Token: | Latin|
Top 7th token. Logit: 13.14 Prob:  1.37% Token: | Greece|
Top 8th token. Logit: 13.11 Prob:  1.33% Token: | Spain|
Top 9th token. Logit: 13.03 Prob:  1.23% Token: | Sicily|


Ranks of the answer tokens: [(' Italy', 0)]

In [ ]:
example_prompt = "Madrid is the capital of"
example_answer = " Spain"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Mad', 'rid', ' is', ' the', ' capital', ' of']
Tokenized answer: [' Spain']


Performance on answer token:
Rank: 0        Logit: 16.77 Prob: 26.98% Token: | Spain|

Top 0th token. Logit: 16.77 Prob: 26.98% Token: | Spain|
Top 1th token. Logit: 16.62 Prob: 23.12% Token: | the|
Top 2th token. Logit: 15.47 Prob:  7.35% Token: | Catalonia|
Top 3th token. Logit: 15.18 Prob:  5.50% Token: | a|
Top 4th token. Logit: 14.26 Prob:  2.19% Token: | Venezuela|
Top 5th token. Logit: 14.13 Prob:  1.92% Token: | Uruguay|
Top 6th token. Logit: 13.99 Prob:  1.68% Token: | one|
Top 7th token. Logit: 13.90 Prob:  1.52% Token: | Portugal|
Top 8th token. Logit: 13.89 Prob:  1.52% Token: | Latin|
Top 9th token. Logit: 13.82 Prob:  1.41% Token: | Colombia|


Ranks of the answer tokens: [(' Spain', 0)]

In [ ]:
example_prompt = "Rome is the capital of Italy. Is it true?"
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' Is', ' it', ' true', '?']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 6        Logit: 15.13 Prob:  2.47% Token: | Yes|

Top 0th token. Logit: 17.27 Prob: 20.87% Token: |
|
Top 1th token. Logit: 15.86 Prob:  5.13% Token: | Is|
Top 2th token. Logit: 15.69 Prob:  4.33% Token: | It|
Top 3th token. Logit: 15.51 Prob:  3.59% Token: | The|
Top 4th token. Logit: 15.42 Prob:  3.30% Token: | I|
Top 5th token. Logit: 15.13 Prob:  2.47% Token: | And|
Top 6th token. Logit: 15.13 Prob:  2.47% Token: | Yes|
Top 7th token. Logit: 15.13 Prob:  2.45% Token: | No|
Top 8th token. Logit: 14.81 Prob:  1.79% Token: | Or|
Top 9th token. Logit: 14.73 Prob:  1.65% Token: | Well|


Ranks of the answer tokens: [(' Yes', 6)]

In [ ]:
example_prompt = "Rome is the capital of Italy. Is it true? Answer:"
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' Is', ' it', ' true', '?', ' Answer', ':']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 14.65 Prob: 14.10% Token: | Yes|

Top 0th token. Logit: 14.65 Prob: 14.10% Token: | Yes|
Top 1th token. Logit: 14.52 Prob: 12.43% Token: | It|
Top 2th token. Logit: 14.02 Prob:  7.48% Token: | yes|
Top 3th token. Logit: 14.00 Prob:  7.35% Token: | No|
Top 4th token. Logit: 13.22 Prob:  3.36% Token: | The|
Top 5th token. Logit: 13.09 Prob:  2.98% Token: |
|
Top 6th token. Logit: 13.05 Prob:  2.86% Token: | "|
Top 7th token. Logit: 12.70 Prob:  2.01% Token: | I|
Top 8th token. Logit: 12.51 Prob:  1.66% Token: | it|
Top 9th token. Logit: 12.46 Prob:  1.58% Token: | Rome|


Ranks of the answer tokens: [(' Yes', 0)]

Dall'ultimo esempio, la keyword *Answer* sembra attivare l'MLP: la conoscenza infatti non puo' derivare dal contesto (perche' non contiene la risposta).

Provo con un contro-esempio, cercando come risposta il *No*:

In [ ]:
example_prompt = "Rome is the capital of Spain. Is it true? Answer:"
example_answer = " No"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Spain', '.', ' Is', ' it', ' true', '?', ' Answer', ':']
Tokenized answer: [' No']


Performance on answer token:
Rank: 2        Logit: 14.11 Prob:  8.21% Token: | No|

Top 0th token. Logit: 14.83 Prob: 16.77% Token: | Yes|
Top 1th token. Logit: 14.45 Prob: 11.49% Token: | It|
Top 2th token. Logit: 14.11 Prob:  8.21% Token: | No|
Top 3th token. Logit: 14.07 Prob:  7.91% Token: | yes|
Top 4th token. Logit: 13.13 Prob:  3.06% Token: | The|
Top 5th token. Logit: 13.06 Prob:  2.87% Token: |
|
Top 6th token. Logit: 12.97 Prob:  2.62% Token: | "|
Top 7th token. Logit: 12.67 Prob:  1.94% Token: | it|
Top 8th token. Logit: 12.63 Prob:  1.86% Token: | I|
Top 9th token. Logit: 12.51 Prob:  1.65% Token: | Of|


Ranks of the answer tokens: [(' No', 2)]

Alla luce del contro-esempio, sembra in realta' che la kewyword *Answer* serva soltanto a potenziare il *Yes* come risposta (e non a sfruttare la conoscenza pregressa).

*Una dimostrazione a favore di questa tesi la si puo' trovare piu' avanti, nella sezione 5-shot di Paris-London-Moscow.*

In [ ]:
example_prompt = "Rome is the capital of Italy. [Yes/No]"
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' [', 'Yes', '/', 'No', ']']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 55       Logit: 10.23 Prob:  0.09% Token: | Yes|

Top 0th token. Logit: 16.60 Prob: 53.59% Token: |
|
Top 1th token. Logit: 14.33 Prob:  5.55% Token: | Rome|
Top 2th token. Logit: 13.99 Prob:  3.95% Token: | The|
Top 3th token. Logit: 13.55 Prob:  2.53% Token: | It|
Top 4th token. Logit: 13.38 Prob:  2.13% Token: |

|
Top 5th token. Logit: 12.86 Prob:  1.27% Token: | Italy|
Top 6th token. Logit: 12.82 Prob:  1.22% Token: |<|endoftext|>|
Top 7th token. Logit: 12.73 Prob:  1.11% Token: | [|
Top 8th token. Logit: 12.59 Prob:  0.97% Token: | In|
Top 9th token. Logit: 12.40 Prob:  0.80% Token: | This|


Ranks of the answer tokens: [(' Yes', 55)]

In [ ]:
example_prompt = "Rome is the capital of Italy. [Yes/No] Answer:"
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' [', 'Yes', '/', 'No', ']', ' Answer', ':']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 13.35 Prob:  6.94% Token: | Yes|

Top 0th token. Logit: 13.36 Prob:  7.05% Token: |
|
Top 1th token. Logit: 13.35 Prob:  6.94% Token: | Yes|
Top 2th token. Logit: 13.27 Prob:  6.42% Token: | The|
Top 3th token. Logit: 12.79 Prob:  3.98% Token: | Rome|
Top 4th token. Logit: 12.69 Prob:  3.60% Token: | Italy|
Top 5th token. Logit: 12.69 Prob:  3.59% Token: | It|
Top 6th token. Logit: 12.33 Prob:  2.52% Token: | No|
Top 7th token. Logit: 12.23 Prob:  2.28% Token: | I|
Top 8th token. Logit: 11.75 Prob:  1.41% Token: | This|
Top 9th token. Logit: 11.71 Prob:  1.35% Token: | A|


Ranks of the answer tokens: [(' Yes', 1)]

#### Few-shot yes/no

In [ ]:
example_prompt = "Madrid is the capital of Spain. Is it true? Yes. Rome is the capital of Italy. Is it true?"
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Mad', 'rid', ' is', ' the', ' capital', ' of', ' Spain', '.', ' Is', ' it', ' true', '?', ' Yes', '.', ' Rome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' Is', ' it', ' true', '?']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 19.83 Prob: 57.96% Token: | Yes|

Top 0th token. Logit: 19.83 Prob: 57.96% Token: | Yes|
Top 1th token. Logit: 18.82 Prob: 21.21% Token: | No|
Top 2th token. Logit: 16.68 Prob:  2.50% Token: | It|
Top 3th token. Logit: 16.15 Prob:  1.47% Token: | Not|
Top 4th token. Logit: 15.99 Prob:  1.24% Token: | Absolutely|
Top 5th token. Logit: 15.82 Prob:  1.05% Token: | The|
Top 6th token. Logit: 15.56 Prob:  0.81% Token: | I|
Top 7th token. Logit: 15.04 Prob:  0.48% Token: | True|
Top 8th token. Logit: 14.95 Prob:  0.44% Token: | In|
Top 9th token. Logit: 14.73 Prob:  0.35% Token: | Of|


Ranks of the answer tokens: [(' Yes', 0)]

In [ ]:
example_prompt = "Madrid is the capital of Spain. [Yes/No] Yes. Rome is the capital of Italy. [Yes/No]"
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Mad', 'rid', ' is', ' the', ' capital', ' of', ' Spain', '.', ' [', 'Yes', '/', 'No', ']', ' Yes', '.', ' Rome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' [', 'Yes', '/', 'No', ']']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 19.11 Prob: 74.37% Token: | Yes|

Top 0th token. Logit: 19.11 Prob: 74.37% Token: | Yes|
Top 1th token. Logit: 17.19 Prob: 10.97% Token: | No|
Top 2th token. Logit: 15.29 Prob:  1.63% Token: |
|
Top 3th token. Logit: 15.06 Prob:  1.30% Token: | Rome|
Top 4th token. Logit: 14.58 Prob:  0.80% Token: | Italy|
Top 5th token. Logit: 14.31 Prob:  0.61% Token: | [|
Top 6th token. Logit: 14.11 Prob:  0.50% Token: | The|
Top 7th token. Logit: 13.88 Prob:  0.40% Token: | I|
Top 8th token. Logit: 13.31 Prob:  0.23% Token: | It|
Top 9th token. Logit: 13.21 Prob:  0.21% Token: | Not|


Ranks of the answer tokens: [(' Yes', 0)]

In [ ]:
example_prompt = ("[Madrid is the capital of Spain. (Yes/No) Yes]"
                  " [Rome is the capital of Italy. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'Mad', 'rid', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', ' [', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 18.74 Prob: 82.81% Token: | Yes|

Top 0th token. Logit: 18.74 Prob: 82.81% Token: | Yes|
Top 1th token. Logit: 16.59 Prob:  9.58% Token: | No|
Top 2th token. Logit: 14.43 Prob:  1.10% Token: | [|
Top 3th token. Logit: 14.25 Prob:  0.93% Token: |]|
Top 4th token. Logit: 13.05 Prob:  0.28% Token: |
|
Top 5th token. Logit: 12.68 Prob:  0.19% Token: | yes|
Top 6th token. Logit: 12.47 Prob:  0.16% Token: | (|
Top 7th token. Logit: 12.46 Prob:  0.15% Token: | Not|
Top 8th token. Logit: 12.01 Prob:  0.10% Token: | The|
Top 9th token. Logit: 11.99 Prob:  0.10% Token: | ]|


Ranks of the answer tokens: [(' Yes', 0)]

In [ ]:
example_prompt = ("[Madrid is the capital of Spain. (Yes/No) Yes],"
                  " [Rome is the capital of Italy. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'Mad', 'rid', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', '],', ' [', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 17.67 Prob: 79.40% Token: | Yes|

Top 0th token. Logit: 17.67 Prob: 79.40% Token: | Yes|
Top 1th token. Logit: 15.21 Prob:  6.74% Token: | No|
Top 2th token. Logit: 14.38 Prob:  2.94% Token: |]|
Top 3th token. Logit: 13.52 Prob:  1.25% Token: | [|
Top 4th token. Logit: 13.41 Prob:  1.11% Token: |],|
Top 5th token. Logit: 13.00 Prob:  0.74% Token: |].|
Top 6th token. Logit: 11.97 Prob:  0.27% Token: | yes|
Top 7th token. Logit: 11.91 Prob:  0.25% Token: | ]|
Top 8th token. Logit: 11.72 Prob:  0.21% Token: |])|
Top 9th token. Logit: 11.65 Prob:  0.19% Token: |
|


Ranks of the answer tokens: [(' Yes', 0)]

In [ ]:
example_prompt = ("Madrid is the capital of Italy. [Yes/No] No.\n"
                  "Rome is the capital of Italy. [Yes/No]")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Mad', 'rid', ' is', ' the', ' capital', ' of', ' Italy', '.', ' [', 'Yes', '/', 'No', ']', ' No', '.', '\n', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' [', 'Yes', '/', 'No', ']']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 16.71 Prob: 10.93% Token: | Yes|

Top 0th token. Logit: 18.72 Prob: 81.42% Token: | No|
Top 1th token. Logit: 16.71 Prob: 10.93% Token: | Yes|
Top 2th token. Logit: 14.57 Prob:  1.28% Token: |
|
Top 3th token. Logit: 13.74 Prob:  0.56% Token: | [|
Top 4th token. Logit: 12.78 Prob:  0.21% Token: | Not|
Top 5th token. Logit: 12.63 Prob:  0.18% Token: | The|
Top 6th token. Logit: 12.16 Prob:  0.12% Token: | NO|
Top 7th token. Logit: 11.82 Prob:  0.08% Token: | (|
Top 8th token. Logit: 11.69 Prob:  0.07% Token: |

|
Top 9th token. Logit: 11.65 Prob:  0.07% Token: | This|


Ranks of the answer tokens: [(' Yes', 1)]

In [ ]:
example_prompt = ("[Madrid is the capital of Italy. (Yes/No) No]\n"
                  "[Rome is the capital of Italy. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'Mad', 'rid', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'R', 'ome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 16.88 Prob: 10.75% Token: | Yes|

Top 0th token. Logit: 18.92 Prob: 82.44% Token: | No|
Top 1th token. Logit: 16.88 Prob: 10.75% Token: | Yes|
Top 2th token. Logit: 13.76 Prob:  0.47% Token: | [|
Top 3th token. Logit: 13.76 Prob:  0.47% Token: |
|
Top 4th token. Logit: 13.43 Prob:  0.34% Token: | Not|
Top 5th token. Logit: 13.21 Prob:  0.27% Token: |]|
Top 6th token. Logit: 13.03 Prob:  0.23% Token: | (|
Top 7th token. Logit: 12.60 Prob:  0.15% Token: | The|
Top 8th token. Logit: 12.59 Prob:  0.15% Token: | NO|
Top 9th token. Logit: 12.17 Prob:  0.10% Token: | no|


Ranks of the answer tokens: [(' Yes', 1)]

Rome in Italy -> No

In [ ]:
example_prompt = ("[ Madrid is the capital of Italy. (Yes/No) No]\n"
                  "[ Madrid is the capital of Spain. (Yes/No) Yes]\n"
                  "[ Rome is the capital of Italy. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', ' Madrid', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', ' Madrid', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', ' Rome', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 18.76 Prob: 34.10% Token: | Yes|

Top 0th token. Logit: 19.37 Prob: 62.41% Token: | No|
Top 1th token. Logit: 18.76 Prob: 34.10% Token: | Yes|
Top 2th token. Logit: 13.24 Prob:  0.14% Token: | no|
Top 3th token. Logit: 13.19 Prob:  0.13% Token: |
|
Top 4th token. Logit: 13.18 Prob:  0.13% Token: | yes|
Top 5th token. Logit: 13.18 Prob:  0.13% Token: | Not|
Top 6th token. Logit: 12.98 Prob:  0.11% Token: | (|
Top 7th token. Logit: 12.96 Prob:  0.10% Token: | NO|
Top 8th token. Logit: 12.53 Prob:  0.07% Token: | The|
Top 9th token. Logit: 12.32 Prob:  0.05% Token: | A|


Ranks of the answer tokens: [(' Yes', 1)]

Rome in France -> No

In [ ]:
example_prompt = ("[ Madrid is the capital of Italy. (Yes/No) No]\n"
                  "[ Madrid is the capital of Spain. (Yes/No) Yes]\n"
                  "[ Rome is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', ' Madrid', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', ' Madrid', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', ' Rome', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 18.42 Prob: 29.81% Token: | Yes|

Top 0th token. Logit: 19.22 Prob: 66.56% Token: | No|
Top 1th token. Logit: 18.42 Prob: 29.81% Token: | Yes|
Top 2th token. Logit: 13.41 Prob:  0.20% Token: |
|
Top 3th token. Logit: 13.16 Prob:  0.15% Token: | no|
Top 4th token. Logit: 12.98 Prob:  0.13% Token: | yes|
Top 5th token. Logit: 12.91 Prob:  0.12% Token: | Not|
Top 6th token. Logit: 12.81 Prob:  0.11% Token: | NO|
Top 7th token. Logit: 12.81 Prob:  0.11% Token: | (|
Top 8th token. Logit: 12.56 Prob:  0.09% Token: | [|
Top 9th token. Logit: 12.19 Prob:  0.06% Token: | The|


Ranks of the answer tokens: [(' Yes', 1)]

Dai due precendenti sembra rispondere senza capire. Forse per rispettare la sequenza alternata delle risposte: *no, yes -> no*.

Da provare con piu' esempi.

### Cambio delle citta'

`model.to_str_tokens` splits a string into the tokens *as a list of substrings*, and so lets you explore what the text looks like.

In [ ]:
print(model.to_str_tokens("Rome"))

['<|endoftext|>', 'R', 'ome']


In [ ]:
print(model.to_str_tokens("Madrid"))

['<|endoftext|>', 'Mad', 'rid']


Sostituti (non divisi):

In [ ]:
print(model.to_str_tokens("Paris"))

['<|endoftext|>', 'Paris']


In [ ]:
print(model.to_str_tokens("France"))

['<|endoftext|>', 'France']


In [ ]:
print(model.to_str_tokens("London"))

['<|endoftext|>', 'London']


In [ ]:
print(model.to_str_tokens("UK"))

['<|endoftext|>', 'UK']


In [ ]:
print(model.to_str_tokens("Moscow"))

['<|endoftext|>', 'Moscow']


In [ ]:
print(model.to_str_tokens("Russia"))

['<|endoftext|>', 'Russia']


### Verifico la conoscenza del modello

#### Completing sentence

In [ ]:
example_prompt = "Paris is the capital of"
example_answer = " France"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Paris', ' is', ' the', ' capital', ' of']
Tokenized answer: [' France']


Performance on answer token:
Rank: 0        Logit: 16.93 Prob: 33.42% Token: | France|

Top 0th token. Logit: 16.93 Prob: 33.42% Token: | France|
Top 1th token. Logit: 16.61 Prob: 24.46% Token: | the|
Top 2th token. Logit: 15.30 Prob:  6.59% Token: | a|
Top 3th token. Logit: 14.68 Prob:  3.52% Token: | Europe|
Top 4th token. Logit: 13.98 Prob:  1.75% Token: | French|
Top 5th token. Logit: 13.93 Prob:  1.67% Token: | Paris|
Top 6th token. Logit: 13.78 Prob:  1.43% Token: | Belgium|
Top 7th token. Logit: 13.75 Prob:  1.39% Token: | one|
Top 8th token. Logit: 13.22 Prob:  0.82% Token: | an|
Top 9th token. Logit: 12.86 Prob:  0.57% Token: | Germany|


Ranks of the answer tokens: [(' France', 0)]

In [ ]:
example_prompt = "London is the capital of the"
example_answer = " UK"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'London', ' is', ' the', ' capital', ' of', ' the']
Tokenized answer: [' UK']


Performance on answer token:
Rank: 1        Logit: 14.60 Prob: 12.37% Token: | UK|

Top 0th token. Logit: 14.77 Prob: 14.68% Token: | world|
Top 1th token. Logit: 14.60 Prob: 12.37% Token: | UK|
Top 2th token. Logit: 14.03 Prob:  7.03% Token: | European|
Top 3th token. Logit: 14.01 Prob:  6.91% Token: | United|
Top 4th token. Logit: 13.63 Prob:  4.69% Token: | British|
Top 5th token. Logit: 13.41 Prob:  3.79% Token: | EU|
Top 6th token. Logit: 12.08 Prob:  1.00% Token: | West|
Top 7th token. Logit: 12.08 Prob:  1.00% Token: | U|
Top 8th token. Logit: 11.98 Prob:  0.90% Token: | country|
Top 9th token. Logit: 11.93 Prob:  0.86% Token: | "|


Ranks of the answer tokens: [(' UK', 1)]

Nota: Sostituire, nel resto del notebook, *of UK* con *of the UK*.

In [ ]:
example_prompt = "Moscow is the capital of"
example_answer = " Russia"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Moscow', ' is', ' the', ' capital', ' of']
Tokenized answer: [' Russia']


Performance on answer token:
Rank: 1        Logit: 16.39 Prob: 20.18% Token: | Russia|

Top 0th token. Logit: 16.73 Prob: 28.47% Token: | the|
Top 1th token. Logit: 16.39 Prob: 20.18% Token: | Russia|
Top 2th token. Logit: 15.55 Prob:  8.67% Token: | a|
Top 3th token. Logit: 15.20 Prob:  6.15% Token: | Ukraine|
Top 4th token. Logit: 14.26 Prob:  2.39% Token: | Belarus|
Top 5th token. Logit: 14.14 Prob:  2.13% Token: | Kazakhstan|
Top 6th token. Logit: 13.77 Prob:  1.47% Token: | an|
Top 7th token. Logit: 13.63 Prob:  1.28% Token: | Crimea|
Top 8th token. Logit: 13.32 Prob:  0.93% Token: | one|
Top 9th token. Logit: 13.32 Prob:  0.93% Token: | Syria|


Ranks of the answer tokens: [(' Russia', 1)]

Faccio delle contro-prove, negando le frasi:

In [ ]:
example_prompt = "Paris is not the capital of"
example_answer = " France"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Paris', ' is', ' not', ' the', ' capital', ' of']
Tokenized answer: [' France']


Performance on answer token:
Rank: 1        Logit: 16.42 Prob: 25.12% Token: | France|

Top 0th token. Logit: 16.43 Prob: 25.17% Token: | the|
Top 1th token. Logit: 16.42 Prob: 25.12% Token: | France|
Top 2th token. Logit: 15.59 Prob: 10.88% Token: | Europe|
Top 3th token. Logit: 15.52 Prob: 10.17% Token: | a|
Top 4th token. Logit: 13.90 Prob:  2.01% Token: | Paris|
Top 5th token. Logit: 13.56 Prob:  1.43% Token: | an|
Top 6th token. Logit: 13.25 Prob:  1.05% Token: | French|
Top 7th token. Logit: 12.98 Prob:  0.80% Token: | Belgium|
Top 8th token. Logit: 12.95 Prob:  0.78% Token: | one|
Top 9th token. Logit: 12.67 Prob:  0.59% Token: | Germany|


Ranks of the answer tokens: [(' France', 1)]

In [ ]:
example_prompt = "London is not the capital of the"
example_answer = " UK"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'London', ' is', ' not', ' the', ' capital', ' of', ' the']
Tokenized answer: [' UK']


Performance on answer token:
Rank: 2        Logit: 14.69 Prob:  6.27% Token: | UK|

Top 0th token. Logit: 16.68 Prob: 46.04% Token: | world|
Top 1th token. Logit: 14.70 Prob:  6.36% Token: | EU|
Top 2th token. Logit: 14.69 Prob:  6.27% Token: | UK|
Top 3th token. Logit: 14.46 Prob:  4.99% Token: | United|
Top 4th token. Logit: 14.42 Prob:  4.82% Token: | European|
Top 5th token. Logit: 13.39 Prob:  1.71% Token: | British|
Top 6th token. Logit: 12.85 Prob:  1.00% Token: | West|
Top 7th token. Logit: 12.82 Prob:  0.97% Token: | euro|
Top 8th token. Logit: 12.79 Prob:  0.94% Token: | eurozone|
Top 9th token. Logit: 12.70 Prob:  0.86% Token: | U|


Ranks of the answer tokens: [(' UK', 2)]

In [ ]:
example_prompt = "Moscow is not the capital of"
example_answer = " Russia"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', 'Moscow', ' is', ' not', ' the', ' capital', ' of']
Tokenized answer: [' Russia']


Performance on answer token:
Rank: 2        Logit: 15.77 Prob: 13.31% Token: | Russia|

Top 0th token. Logit: 16.58 Prob: 29.84% Token: | the|
Top 1th token. Logit: 15.93 Prob: 15.54% Token: | a|
Top 2th token. Logit: 15.77 Prob: 13.31% Token: | Russia|
Top 3th token. Logit: 14.83 Prob:  5.17% Token: | Ukraine|
Top 4th token. Logit: 14.31 Prob:  3.10% Token: | an|
Top 5th token. Logit: 14.21 Prob:  2.80% Token: | Europe|
Top 6th token. Logit: 13.41 Prob:  1.25% Token: | Moscow|
Top 7th token. Logit: 13.14 Prob:  0.96% Token: | one|
Top 8th token. Logit: 13.12 Prob:  0.94% Token: | any|
Top 9th token. Logit: 13.06 Prob:  0.89% Token: | Syria|


Ranks of the answer tokens: [(' Russia', 2)]

Sembra che non ci sia una vera conoscenza appresa da parte del modello riguardo le capitali.

Oppure semplicemente non comprende il fatto che *not* indica una negazione?

*Effettivamente, anche gli esempi nella sezione Apple tree vanno in questa direzione: il not non influisce molto sull'output.*

#### Generating Text

*TransformerLens also has basic text generation functionality, which can be useful for generally exploring what the model is capable of (thanks to Ansh Radhakrishnan for adding this!). This is pretty rough functionality, and where possible I recommend using more established libraries like HuggingFace for this.*

In [ ]:
# NBVAL_IGNORE_OUTPUT
model.generate("Paris is the capital of", max_new_tokens=50, temperature=0.7, prepend_bos=True)

  0%|          | 0/50 [00:00<?, ?it/s]

'Paris is the capital of France and one of the most important cities in the world, and it is mesmerizing to watch the sights, sounds, and life of the city.\n\nParis is also a popular destination for travelers looking to get to the United States or Europe for'

In [ ]:
# NBVAL_IGNORE_OUTPUT
model.generate("London is the capital of", max_new_tokens=50, temperature=0.7, prepend_bos=True)

  0%|          | 0/50 [00:00<?, ?it/s]

"London is the capital of the UK and the second largest city in the world after some of the world's major cities. The capital of Europe includes Poland, Monaco, Slovakia, and Hungary. The capital is Belfast, Ireland. The capital of Switzerland is Zurich.\n\nThe"

In [ ]:
# NBVAL_IGNORE_OUTPUT
model.generate("Moscow is the capital of", max_new_tokens=50, temperature=0.7, prepend_bos=True)

  0%|          | 0/50 [00:00<?, ?it/s]

'Moscow is the capital of Russia and has been an important market for the Russian government since the 1990s. Russia has for decades been a center of international energy cooperation. In 2010, Russia\'s government welcomed the United States after signing a "more than 200 billion rubles" trade'

Faccio delle contro-prove, negando le frasi dei prompt:

In [ ]:
# NBVAL_IGNORE_OUTPUT
model.generate("Paris is not the capital of", max_new_tokens=50, temperature=0.7, prepend_bos=True)

  0%|          | 0/50 [00:00<?, ?it/s]

'Paris is not the capital of the world.\n\nThe capital of France is Paris.\n\nParis is the capital of the world.\n\nThe capital of France is France.\n\nFrance and France are two countries.\n\nFrance is the capital of the world.'

In [ ]:
# NBVAL_IGNORE_OUTPUT
model.generate("London is not the capital of", max_new_tokens=50, temperature=0.7, prepend_bos=True)

  0%|          | 0/50 [00:00<?, ?it/s]

"London is not the capital of the world's biggest economy. It has become a hub of financial services. Now, London's banks may have to face up to that fact, because there are no plans for London's housing market to grow much more than it has.\n\nThe"

In [ ]:
# NBVAL_IGNORE_OUTPUT
model.generate("Moscow is not the capital of", max_new_tokens=50, temperature=0.7, prepend_bos=True)

  0%|          | 0/50 [00:00<?, ?it/s]

'Moscow is not the capital of the Russian Federation, but rather its police and military. It\'s also not the largest country in the Russian Federation.\n\n"Russia\'s police and military are on a mission to protect our citizens, their families and the nation. One of the only'

### Paris - London - Moscow

#### 1-shot (*si allinea alla risposta piu' frequente, ie l'unica*)

yes -> yes

In [ ]:
example_prompt = ("[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 19.39 Prob: 87.05% Token: | Yes|

Top 0th token. Logit: 19.39 Prob: 87.05% Token: | Yes|
Top 1th token. Logit: 17.13 Prob:  9.08% Token: | No|
Top 2th token. Logit: 13.64 Prob:  0.28% Token: |
|
Top 3th token. Logit: 13.03 Prob:  0.15% Token: | [|
Top 4th token. Logit: 12.90 Prob:  0.13% Token: | yes|
Top 5th token. Logit: 12.89 Prob:  0.13% Token: | Not|
Top 6th token. Logit: 12.41 Prob:  0.08% Token: |Yes|
Top 7th token. Logit: 12.35 Prob:  0.08% Token: | YES|
Top 8th token. Logit: 12.16 Prob:  0.06% Token: | The|
Top 9th token. Logit: 12.15 Prob:  0.06% Token: | (|


Ranks of the answer tokens: [(' Yes', 0)]

no -> no

In [ ]:
example_prompt = ("[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 17.73 Prob: 13.26% Token: | Yes|

Top 0th token. Logit: 19.56 Prob: 82.80% Token: | No|
Top 1th token. Logit: 17.73 Prob: 13.26% Token: | Yes|
Top 2th token. Logit: 13.66 Prob:  0.23% Token: | Not|
Top 3th token. Logit: 13.60 Prob:  0.21% Token: |
|
Top 4th token. Logit: 13.45 Prob:  0.18% Token: | [|
Top 5th token. Logit: 13.14 Prob:  0.13% Token: | NO|
Top 6th token. Logit: 12.84 Prob:  0.10% Token: | (|
Top 7th token. Logit: 12.70 Prob:  0.09% Token: | no|
Top 8th token. Logit: 12.65 Prob:  0.08% Token: | The|
Top 9th token. Logit: 12.23 Prob:  0.05% Token: |No|


Ranks of the answer tokens: [(' Yes', 1)]

Riflessione: Perche' il *Yes* ha una percentuale maggiore rispetto al *No* (facendo il paragone con l'esempio precedente)? Viene potenziato dalla conoscenza pregressa?

*Si potrebbe verificare con le MCQs: dal momento che le alternative sono maggiori, se le risposte sbagliate avessero tutte la stessa percentuale, mentre quella esatta fosse l'unica ad avere una percentuale maggiore, allora andrebbe a favore della tesi secondo cui la conoscenza pregressa contribuisce a potenziare la scelta della risposta corretta.*

#### 2-shot (*prosegue la sequenza alternata*)

Paris in France -> No

*no, yes -> no*

In [ ]:
example_prompt = ("[London is the capital of Italy. (Yes/No) No]\n"
                  "[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 18.59 Prob: 33.89% Token: | Yes|

Top 0th token. Logit: 19.20 Prob: 62.52% Token: | No|
Top 1th token. Logit: 18.59 Prob: 33.89% Token: | Yes|
Top 2th token. Logit: 13.27 Prob:  0.17% Token: | no|
Top 3th token. Logit: 13.18 Prob:  0.15% Token: | Not|
Top 4th token. Logit: 13.13 Prob:  0.14% Token: |
|
Top 5th token. Logit: 13.06 Prob:  0.13% Token: | yes|
Top 6th token. Logit: 13.00 Prob:  0.13% Token: | NO|
Top 7th token. Logit: 12.92 Prob:  0.12% Token: | (|
Top 8th token. Logit: 12.54 Prob:  0.08% Token: | [|
Top 9th token. Logit: 12.29 Prob:  0.06% Token: | A|


Ranks of the answer tokens: [(' Yes', 1)]

Paris in France -> Yes

*yes, no -> yes*

In [ ]:
example_prompt = ("[London is the capital of UK. (Yes/No) Yes]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 18.67 Prob: 59.13% Token: | Yes|

Top 0th token. Logit: 18.67 Prob: 59.13% Token: | Yes|
Top 1th token. Logit: 18.13 Prob: 34.78% Token: | No|
Top 2th token. Logit: 13.48 Prob:  0.33% Token: | yes|
Top 3th token. Logit: 13.08 Prob:  0.22% Token: |
|
Top 4th token. Logit: 12.81 Prob:  0.17% Token: | (|
Top 5th token. Logit: 12.72 Prob:  0.16% Token: | no|
Top 6th token. Logit: 12.66 Prob:  0.15% Token: | Not|
Top 7th token. Logit: 12.52 Prob:  0.13% Token: | I|
Top 8th token. Logit: 12.42 Prob:  0.11% Token: | NO|
Top 9th token. Logit: 12.40 Prob:  0.11% Token: | A|


Ranks of the answer tokens: [(' Yes', 0)]

**Riflessione**

Ha risposto basandosi solamente sullo schema delle risposte precedenti (alternate).

Si puo' pensare che metta in secondo piano la sua conoscenza, per rispettare lo schema alternato di risposte?

#### 3-shot (*si allinea alla risposta piu' frequente*)

*2 yes, 1 no -> yes*

In [ ]:
example_prompt = ("[London is the capital of Italy. (Yes/No) No]\n"
                  "[Moscow is the capital of Russia. (Yes/No) Yes]\n"
                  "[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 19.58 Prob: 55.21% Token: | Yes|

Top 0th token. Logit: 19.58 Prob: 55.21% Token: | Yes|
Top 1th token. Logit: 19.31 Prob: 42.32% Token: | No|
Top 2th token. Logit: 13.79 Prob:  0.17% Token: | yes|
Top 3th token. Logit: 13.36 Prob:  0.11% Token: | no|
Top 4th token. Logit: 13.10 Prob:  0.08% Token: | Not|
Top 5th token. Logit: 13.09 Prob:  0.08% Token: | NO|
Top 6th token. Logit: 13.01 Prob:  0.08% Token: | (|
Top 7th token. Logit: 12.97 Prob:  0.07% Token: |
|
Top 8th token. Logit: 12.73 Prob:  0.06% Token: | [|
Top 9th token. Logit: 12.53 Prob:  0.05% Token: | YES|


Ranks of the answer tokens: [(' Yes', 0)]

*2 yes, 1 no -> yes*

In [ ]:
example_prompt = ("[Moscow is the capital of Russia. (Yes/No) Yes]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 19.09 Prob: 57.00% Token: | Yes|

Top 0th token. Logit: 19.09 Prob: 57.00% Token: | Yes|
Top 1th token. Logit: 18.72 Prob: 39.36% Token: | No|
Top 2th token. Logit: 14.00 Prob:  0.35% Token: | yes|
Top 3th token. Logit: 13.19 Prob:  0.16% Token: | no|
Top 4th token. Logit: 12.91 Prob:  0.12% Token: |
|
Top 5th token. Logit: 12.88 Prob:  0.11% Token: | (|
Top 6th token. Logit: 12.84 Prob:  0.11% Token: | NO|
Top 7th token. Logit: 12.68 Prob:  0.09% Token: | [|
Top 8th token. Logit: 12.53 Prob:  0.08% Token: | Not|
Top 9th token. Logit: 12.50 Prob:  0.08% Token: | YES|


Ranks of the answer tokens: [(' Yes', 0)]

*2 no, 1 yes -> no*

In [ ]:
example_prompt = ("[Moscow is the capital of Italy. (Yes/No) No]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 18.44 Prob: 33.47% Token: | Yes|

Top 0th token. Logit: 19.07 Prob: 62.79% Token: | No|
Top 1th token. Logit: 18.44 Prob: 33.47% Token: | Yes|
Top 2th token. Logit: 13.21 Prob:  0.18% Token: | no|
Top 3th token. Logit: 13.15 Prob:  0.17% Token: | (|
Top 4th token. Logit: 13.10 Prob:  0.16% Token: | yes|
Top 5th token. Logit: 12.91 Prob:  0.13% Token: | NO|
Top 6th token. Logit: 12.87 Prob:  0.13% Token: |
|
Top 7th token. Logit: 12.74 Prob:  0.11% Token: | Not|
Top 8th token. Logit: 12.57 Prob:  0.09% Token: | [|
Top 9th token. Logit: 12.10 Prob:  0.06% Token: | N|


Ranks of the answer tokens: [(' Yes', 1)]

#### 4-shot (*si allinea alla risposta piu' frequente, 50-50 se c'e' parita'*)

2 yes, 2 no -> 50-50

In [ ]:
example_prompt = ("[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Moscow is the capital of Russia. (Yes/No) Yes]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 18.83 Prob: 47.48% Token: | Yes|

Top 0th token. Logit: 18.84 Prob: 48.23% Token: | No|
Top 1th token. Logit: 18.83 Prob: 47.48% Token: | Yes|
Top 2th token. Logit: 13.52 Prob:  0.23% Token: | yes|
Top 3th token. Logit: 13.24 Prob:  0.18% Token: | no|
Top 4th token. Logit: 13.10 Prob:  0.15% Token: | (|
Top 5th token. Logit: 13.01 Prob:  0.14% Token: |
|
Top 6th token. Logit: 12.96 Prob:  0.13% Token: | NO|
Top 7th token. Logit: 12.57 Prob:  0.09% Token: | [|
Top 8th token. Logit: 12.48 Prob:  0.08% Token: | Not|
Top 9th token. Logit: 12.40 Prob:  0.08% Token: | YES|


Ranks of the answer tokens: [(' Yes', 1)]

2 yes, 2 no -> 50-50

In [ ]:
example_prompt = ("[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[Moscow is the capital of Russia. (Yes/No) Yes]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 19.23 Prob: 47.23% Token: | Yes|

Top 0th token. Logit: 19.29 Prob: 49.69% Token: | No|
Top 1th token. Logit: 19.23 Prob: 47.23% Token: | Yes|
Top 2th token. Logit: 13.91 Prob:  0.23% Token: | yes|
Top 3th token. Logit: 13.43 Prob:  0.14% Token: | no|
Top 4th token. Logit: 13.10 Prob:  0.10% Token: | NO|
Top 5th token. Logit: 13.06 Prob:  0.10% Token: | (|
Top 6th token. Logit: 13.05 Prob:  0.10% Token: |
|
Top 7th token. Logit: 12.89 Prob:  0.08% Token: | [|
Top 8th token. Logit: 12.63 Prob:  0.06% Token: | Not|
Top 9th token. Logit: 12.59 Prob:  0.06% Token: | YES|


Ranks of the answer tokens: [(' Yes', 1)]

3 no, 1 yes -> No

In [ ]:
example_prompt = ("[London is the capital of Russia. (Yes/No) No]\n"
                  "[Moscow is the capital of Russia. (Yes/No) Yes]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 19.46 Prob: 30.38% Token: | Yes|

Top 0th token. Logit: 20.26 Prob: 67.65% Token: | No|
Top 1th token. Logit: 19.46 Prob: 30.38% Token: | Yes|
Top 2th token. Logit: 13.96 Prob:  0.12% Token: | yes|
Top 3th token. Logit: 13.89 Prob:  0.12% Token: | no|
Top 4th token. Logit: 13.75 Prob:  0.10% Token: | NO|
Top 5th token. Logit: 13.19 Prob:  0.06% Token: | (|
Top 6th token. Logit: 13.17 Prob:  0.06% Token: | Not|
Top 7th token. Logit: 13.00 Prob:  0.05% Token: | [|
Top 8th token. Logit: 12.91 Prob:  0.04% Token: |
|
Top 9th token. Logit: 12.83 Prob:  0.04% Token: | YES|


Ranks of the answer tokens: [(' Yes', 1)]

Fornisco, nel prompt, la soluzione:

In [ ]:
example_prompt = ("[London is the capital of Russia. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No) Yes]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 20.37 Prob: 37.25% Token: | Yes|

Top 0th token. Logit: 20.87 Prob: 61.46% Token: | No|
Top 1th token. Logit: 20.37 Prob: 37.25% Token: | Yes|
Top 2th token. Logit: 14.22 Prob:  0.08% Token: | NO|
Top 3th token. Logit: 14.22 Prob:  0.08% Token: | yes|
Top 4th token. Logit: 14.09 Prob:  0.07% Token: | no|
Top 5th token. Logit: 13.68 Prob:  0.05% Token: | Not|
Top 6th token. Logit: 13.44 Prob:  0.04% Token: | YES|
Top 7th token. Logit: 13.40 Prob:  0.03% Token: | (|
Top 8th token. Logit: 13.27 Prob:  0.03% Token: | [|
Top 9th token. Logit: 12.93 Prob:  0.02% Token: |
|


Ranks of the answer tokens: [(' Yes', 1)]

**Riflessione sull'esempio**

Continua a rispettare lo schema *3 no, 1 yes -> no*, pero' la percentuale del *Yes* e' aumentata.

Questo grazie alla soluzione fornita?

Se si', ha davvero capito o sta solo copiando la sequenza precedente nel contesto?

*Effettivamente, anche gli esempi nella sezione Apple tree vanno in questa direzione: esprimere il concetto stesso, appena prima di chiederlo al modello, non comporta un risultato differente. (Ma aumentano comunque la percentuale del Yes?)*

4 no, 0 yes -> No

In [ ]:
example_prompt = ("[London is the capital of Russia. (Yes/No) No]\n"
                  "[Moscow is the capital of UK. (Yes/No) No]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 19.95 Prob:  3.61% Token: | Yes|

Top 0th token. Logit: 23.23 Prob: 95.82% Token: | No|
Top 1th token. Logit: 19.95 Prob:  3.61% Token: | Yes|
Top 2th token. Logit: 15.38 Prob:  0.04% Token: | NO|
Top 3th token. Logit: 15.37 Prob:  0.04% Token: | no|
Top 4th token. Logit: 15.28 Prob:  0.03% Token: | Not|
Top 5th token. Logit: 14.92 Prob:  0.02% Token: | [|
Top 6th token. Logit: 14.64 Prob:  0.02% Token: | (|
Top 7th token. Logit: 14.51 Prob:  0.02% Token: |
|
Top 8th token. Logit: 14.32 Prob:  0.01% Token: | N|
Top 9th token. Logit: 14.06 Prob:  0.01% Token: | The|


Ranks of the answer tokens: [(' Yes', 1)]

Provo la keyword *Answer*:

In [ ]:
example_prompt = ("[London is the capital of Russia. (Yes/No) No]\n"
                  "[Moscow is the capital of UK. (Yes/No) No]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No) Answer:")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'London', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')', ' Answer', ':']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 0        Logit: 12.51 Prob:  7.39% Token: | Yes|

Top 0th token. Logit: 12.51 Prob:  7.39% Token: | Yes|
Top 1th token. Logit: 12.49 Prob:  7.23% Token: | No|
Top 2th token. Logit: 12.34 Prob:  6.26% Token: | London|
Top 3th token. Logit: 11.66 Prob:  3.17% Token: |
|
Top 4th token. Logit: 11.61 Prob:  3.02% Token: | The|
Top 5th token. Logit: 11.52 Prob:  2.74% Token: | Moscow|
Top 6th token. Logit: 11.35 Prob:  2.31% Token: | yes|
Top 7th token. Logit: 11.19 Prob:  1.97% Token: | "|
Top 8th token. Logit: 11.12 Prob:  1.84% Token: | I|
Top 9th token. Logit: 10.95 Prob:  1.55% Token: | [|


Ranks of the answer tokens: [(' Yes', 0)]

Questo dimostra che la keyword *Answer* serve solo a potenziare il *Yes*.

#### 5-shot (*si allinea alla risposta piu' frequente*)

3 no, 2 yes -> no

In [ ]:
example_prompt = ("[Moscow is the capital of Italy. (Yes/No) No]\n"
                  "[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[Moscow is the capital of Russia. (Yes/No) Yes]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 19.30 Prob: 34.07% Token: | Yes|

Top 0th token. Logit: 19.93 Prob: 63.68% Token: | No|
Top 1th token. Logit: 19.30 Prob: 34.07% Token: | Yes|
Top 2th token. Logit: 13.75 Prob:  0.13% Token: | yes|
Top 3th token. Logit: 13.63 Prob:  0.12% Token: | no|
Top 4th token. Logit: 13.49 Prob:  0.10% Token: | NO|
Top 5th token. Logit: 13.19 Prob:  0.08% Token: |
|
Top 6th token. Logit: 13.06 Prob:  0.07% Token: | (|
Top 7th token. Logit: 12.95 Prob:  0.06% Token: | [|
Top 8th token. Logit: 12.78 Prob:  0.05% Token: | Not|
Top 9th token. Logit: 12.58 Prob:  0.04% Token: | YES|


Ranks of the answer tokens: [(' Yes', 1)]

3 no, 2 yes -> no

In [ ]:
example_prompt = ("[Moscow is the capital of Italy. (Yes/No) No]\n"
                  "[Moscow is the capital of Spain. (Yes/No) No]\n"
                  "[London is the capital of Italy. (Yes/No) No]\n"
                  "[London is the capital of UK. (Yes/No) Yes]\n"
                  "[Moscow is the capital of Russia. (Yes/No) Yes]\n"
                  "[Paris is the capital of France. (Yes/No)")
example_answer = " Yes"
utils.test_prompt(example_prompt, example_answer, model, prepend_bos=True)

Tokenized prompt: ['<|endoftext|>', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Spain', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' Italy', '.', ' (', 'Yes', '/', 'No', ')', ' No', ']', '\n', '[', 'London', ' is', ' the', ' capital', ' of', ' UK', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Moscow', ' is', ' the', ' capital', ' of', ' Russia', '.', ' (', 'Yes', '/', 'No', ')', ' Yes', ']', '\n', '[', 'Paris', ' is', ' the', ' capital', ' of', ' France', '.', ' (', 'Yes', '/', 'No', ')']
Tokenized answer: [' Yes']


Performance on answer token:
Rank: 1        Logit: 18.77 Prob: 36.32% Token: | Yes|

Top 0th token. Logit: 19.28 Prob: 60.58% Token: | No|
Top 1th token. Logit: 18.77 Prob: 36.32% Token: | Yes|
Top 2th token. Logit: 13.38 Prob:  0.17% Token: | no|
Top 3th token. Logit: 13.32 Prob:  0.16% Token: | yes|
Top 4th token. Logit: 13.00 Prob:  0.11% Token: | (|
Top 5th token. Logit: 12.98 Prob:  0.11% Token: |
|
Top 6th token. Logit: 12.86 Prob:  0.10% Token: | NO|
Top 7th token. Logit: 12.65 Prob:  0.08% Token: | [|
Top 8th token. Logit: 12.47 Prob:  0.07% Token: | Not|
Top 9th token. Logit: 12.05 Prob:  0.04% Token: | N|


Ranks of the answer tokens: [(' Yes', 1)]

#### Possibile spiegazione

Viene data maggiore importanza all'ICL piuttosto che alla conoscenza pregressa (del training).

MLP non si attivano (o vengono ignorati) e tutta la conoscenza messa in campo dipende da cio' che le induction heads derivano dal contesto.

Ho cosi' verificato che le induction heads sono il motore principale nell'ICL (come affermano negli articoli di Anthropic).